# Zona Oro — kpi_precios, kpi_sentimiento, clusters_productos
**Proyecto:** food-price-forecasting-peru | Big Data Analytics UNMSM 2025-II | Entregable 5


**Objetivo:** construir las 3 tablas finales que alimentan el dashboard Streamlit, sin ningun calculo adicional de parte de Andrew.

| Tabla | Alimenta | Origen |
|---|---|---|
| kpi_precios | Vista 1 (alertas), Vista 2, Vista 4 | plata/precios_clima_integrado/precios_spark/ |
| kpi_sentimiento | Vista 5 | plata/sentimiento_semanal.parquet |
| clusters_productos | Vista 3 | Resultado en memoria del Modelo 2 (notebook 05) |

In [1]:
# Instalacion (ejecutar una vez por sesion)
!pip install -q pyspark google-cloud-storage

In [37]:
# Variables de configuracion
BUCKET_NAME = "food-price-peru-2025-01"
PROJECT_ID  = "proyecto-food-price"

# Origenes (Plata / Staging)
PATH_PRECIOS_PLATA     = f"gs://{BUCKET_NAME}/plata/precios_clima_integrado/precios_spark/"
PATH_SENTIMIENTO_PLATA = f"gs://{BUCKET_NAME}/plata/tweets_sentimiento_semanal"
PATH_STAGING_CLUSTERS  = f"gs://{BUCKET_NAME}/oro/clustering_productos/"

# Destinos Finales (Zona Oro)
PATH_ORO_PRECIOS     = f"gs://{BUCKET_NAME}/oro/kpi_precios/"
PATH_ORO_SENTIMIENTO = f"gs://{BUCKET_NAME}/oro/kpi_sentimiento/"
PATH_ORO_CLUSTERS    = f"gs://{BUCKET_NAME}/oro/clusters_productos/"

UMBRAL_ALERTA   = 10.0
UMBRAL_ATENCION = 5.0

print("Rutas configuradas correctamente (Staging vs Oro).")

Rutas configuradas correctamente (Staging vs Oro).


In [11]:
# Autenticacion GCP e inicializacion de Spark
import os
from google.colab import auth
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

auth.authenticate_user()
!gcloud config set project {PROJECT_ID}

jar_name = "gcs-connector.jar"
jar_path = os.path.abspath(jar_name)
if not os.path.exists(jar_path):
    !wget -q https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.14/gcs-connector-hadoop3-2.2.14-shaded.jar -O {jar_name}

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {jar_path} pyspark-shell"

spark = (SparkSession.builder
    .appName("D08-ZonaOro")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.driver.extraClassPath", jar_path)
    .config("spark.executor.extraClassPath", jar_path)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark iniciado. Version:", spark.version)

Updated property [core/project].
Spark iniciado. Version: 4.0.3


## Parte 1 — kpi_precios

In [5]:
# Cargar precios_spark y verificar columnas reales
df_precios_raw = spark.read.parquet(PATH_PRECIOS_PLATA)

print("Schema real de precios_spark:")
df_precios_raw.printSchema()
print(f"Total filas: {df_precios_raw.count()}")
df_precios_raw.show(5, truncate=False)

Schema real de precios_spark:
root
 |-- fecha: timestamp (nullable = true)
 |-- precio: double (nullable = true)
 |-- producto: string (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- distrito_lima: string (nullable = true)
 |-- ubigeo_inei: string (nullable = true)
 |-- region_clima_origen: string (nullable = true)
 |-- mes: long (nullable = true)
 |-- semana: long (nullable = true)
 |-- dia_semana: long (nullable = true)
 |-- trimestre: long (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mercado_sisap: string (nullable = true)

Total filas: 34969
+-------------------+------+-------------+---------------+-------------+-----------+-------------------+---+------+----------+---------+----+------------------------------+
|fecha              |precio|producto     |archivo_origen |distrito_lima|ubigeo_inei|region_clima_origen|mes|semana|dia_semana|trimestre|anio|mercado_sisap                 |
+-------------------+------+-------------+---------------+--------

In [46]:
# Definir lista de productos a EXCLUIR (Industriales y Proteinas)
PRODUCTOS_EXCLUIDOS = [
    "Fideos Molitalia Spaghetti / Tallarin",
    "Leche Fresca Gloria En Bolsa ( 12 Bolsas De Litro)",
    "Huevos Rosados",
    "Pollo Vivo"
]

MAPEO_COLUMNAS = {
    "anio": "anio",
    "semana": "semana",
    "producto": "producto",
    "precio": "precio"
}

# Filtrar usando ~F.col(...).isin para excluir los no deseados
df_precios = (df_precios_raw
    .filter(~F.col("producto").isin(PRODUCTOS_EXCLUIDOS))
    .select(
        F.col(MAPEO_COLUMNAS["anio"]).alias("anio"),
        F.col(MAPEO_COLUMNAS["semana"]).alias("semana"),
        F.col(MAPEO_COLUMNAS["producto"]).alias("producto"),
        F.col(MAPEO_COLUMNAS["precio"]).alias("precio")
    )
    .groupBy("producto", "anio", "semana")
    .agg(F.round(F.avg("precio"), 4).alias("precio_promedio"))
)

ventana = Window.partitionBy("producto").orderBy("anio", "semana")

df_kpi_precios = (df_precios
    .withColumn("precio_anterior", F.lag("precio_promedio", 1).over(ventana))
    .withColumn(
        "variacion_porcentual",
        F.when(
            (F.col("precio_anterior").isNotNull()) & (F.col("precio_anterior") != 0),
            F.round(
                (F.col("precio_promedio") - F.col("precio_anterior")) / F.col("precio_anterior") * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "alerta",
        F.when(F.col("variacion_porcentual") >= UMBRAL_ALERTA, "ALERTA")
         .when(F.col("variacion_porcentual") >= UMBRAL_ATENCION, "ATENCION")
         .otherwise("NORMAL")
    )
    .select("anio", "semana", "producto", "precio_promedio", "variacion_porcentual", "alerta")
    .orderBy("producto", F.desc("anio"), F.desc("semana"))
)

print("Vista previa de KPI Precios (Últimas semanas por producto):")
df_kpi_precios.show(15, truncate=False)

Vista previa de KPI Precios (Últimas semanas por producto):
+----+------+--------------------+---------------+--------------------+--------+
|anio|semana|producto            |precio_promedio|variacion_porcentual|alerta  |
+----+------+--------------------+---------------+--------------------+--------+
|2025|52    |Ajo Criollo O Napuri|4.4857         |0.83                |NORMAL  |
|2025|51    |Ajo Criollo O Napuri|4.4486         |-8.49               |NORMAL  |
|2025|50    |Ajo Criollo O Napuri|4.8614         |-13.5               |NORMAL  |
|2025|49    |Ajo Criollo O Napuri|5.62           |1.87                |NORMAL  |
|2025|48    |Ajo Criollo O Napuri|5.5171         |-10.23              |NORMAL  |
|2025|47    |Ajo Criollo O Napuri|6.1457         |-4.23               |NORMAL  |
|2025|46    |Ajo Criollo O Napuri|6.4171         |6.95                |ATENCION|
|2025|45    |Ajo Criollo O Napuri|6.0            |6.93                |ATENCION|
|2025|44    |Ajo Criollo O Napuri|5.6114         

In [47]:
# Guardar kpi_precios en Zona Oro y verificar
df_kpi_precios.write.mode("overwrite").parquet(PATH_ORO_PRECIOS)
print(f"Guardado: {PATH_ORO_PRECIOS}")

check1 = spark.read.parquet(PATH_ORO_PRECIOS)
print(f"Filas verificadas: {check1.count()}")
check1.printSchema()

Guardado: gs://food-price-peru-2025-01/oro/kpi_precios/
Filas verificadas: 3132
root
 |-- anio: integer (nullable = true)
 |-- semana: long (nullable = true)
 |-- producto: string (nullable = true)
 |-- precio_promedio: double (nullable = true)
 |-- variacion_porcentual: double (nullable = true)
 |-- alerta: string (nullable = true)



## Parte 2 — kpi_sentimiento

In [17]:
# Cargar y verificar columnas de sentimiento_semanal
df_sentimiento_raw = spark.read.parquet(PATH_SENTIMIENTO_PLATA)

print("Schema real de sentimiento_semanal:")
df_sentimiento_raw.printSchema()
df_sentimiento_raw.show(5, truncate=False)

Schema real de sentimiento_semanal:
root
 |-- semana: timestamp (nullable = true)
 |-- n_tweets: long (nullable = true)
 |-- sentiment_promedio: double (nullable = true)
 |-- n_negativos: long (nullable = true)
 |-- n_positivos: long (nullable = true)
 |-- n_neutros: long (nullable = true)
 |-- total_retweets: long (nullable = true)
 |-- pct_negativo: double (nullable = true)
 |-- pct_positivo: double (nullable = true)

+-------------------+--------+-------------------+-----------+-----------+---------+--------------+------------+------------+
|semana             |n_tweets|sentiment_promedio |n_negativos|n_positivos|n_neutros|total_retweets|pct_negativo|pct_positivo|
+-------------------+--------+-------------------+-----------+-----------+---------+--------------+------------+------------+
|2015-01-05 00:00:00|1       |1.0                |0          |1          |0        |2             |0.0         |100.0       |
|2015-01-12 00:00:00|2       |-0.5               |1          |0         

In [19]:
# Construir kpi_sentimiento

cols_disponibles = df_sentimiento_raw.columns

df_kpi_sentimiento = df_sentimiento_raw

# 1. Calcular porcentaje_negativos si no existe
if "porcentaje_negativos" not in cols_disponibles and "n_negativos" in cols_disponibles:
    df_kpi_sentimiento = df_kpi_sentimiento.withColumn(
        "porcentaje_negativos",
        F.round(F.col("n_negativos") / F.col("n_tweets") * 100, 2)
    )

# 2. Extraer anio y semana de la columna timestamp 'semana'
# y seleccionar columnas finales
df_kpi_sentimiento = (df_kpi_sentimiento
    .withColumn("anio_val", F.year(F.col("semana")))
    .withColumn("semana_val", F.weekofyear(F.col("semana")))
    .select(
        F.col("anio_val").cast("int").alias("anio"),
        F.col("semana_val").cast("int").alias("semana"),
        F.round(F.col("sentiment_promedio"), 4).alias("sentiment_promedio"),
        F.col("n_tweets").cast("long").alias("n_tweets"),
        F.col("pct_negativo").cast("double").alias("porcentaje_negativos")
    )
    .orderBy("anio", "semana")
)

df_kpi_sentimiento.show(10, truncate=False)

+----+------+------------------+--------+--------------------+
|anio|semana|sentiment_promedio|n_tweets|porcentaje_negativos|
+----+------+------------------+--------+--------------------+
|2015|2     |1.0               |1       |0.0                 |
|2015|3     |-0.5              |2       |50.0                |
|2015|4     |-0.6667           |3       |66.67               |
|2015|5     |-0.5              |2       |50.0                |
|2015|6     |-1.0              |1       |100.0               |
|2015|7     |0.0               |1       |0.0                 |
|2015|8     |0.0               |1       |0.0                 |
|2015|10    |0.0               |1       |0.0                 |
|2015|11    |-1.0              |6       |100.0               |
|2015|12    |0.0               |1       |0.0                 |
+----+------+------------------+--------+--------------------+
only showing top 10 rows


In [20]:
# Guardar kpi_sentimiento en Zona Oro y verificar
df_kpi_sentimiento.write.mode("overwrite").parquet(PATH_ORO_SENTIMIENTO)
print(f"Guardado: {PATH_ORO_SENTIMIENTO}")

check2 = spark.read.parquet(PATH_ORO_SENTIMIENTO)
print(f"Filas verificadas: {check2.count()}")
check2.printSchema()

Guardado: gs://food-price-peru-2025-01/oro/kpi_sentimiento/
Filas verificadas: 465
root
 |-- anio: integer (nullable = true)
 |-- semana: integer (nullable = true)
 |-- sentiment_promedio: double (nullable = true)
 |-- n_tweets: long (nullable = true)
 |-- porcentaje_negativos: double (nullable = true)



## Parte 3 — clusters_productos

In [48]:
# Cargar clusters desde el origen (staging)
df_clusters_raw = spark.read.parquet(PATH_STAGING_CLUSTERS)

print("Schema original detectado en Staging:")
df_clusters_raw.printSchema()

print("Verificando productos disponibles para filtrar:")
(df_clusters_raw
    .filter(~F.col("producto").isin(PRODUCTOS_EXCLUIDOS))
    .select("producto")
    .distinct()
    .show(truncate=False))

Schema original detectado en Staging:
root
 |-- producto: string (nullable = true)
 |-- cluster: long (nullable = true)
 |-- precio_promedio_historico: double (nullable = true)
 |-- volatilidad: double (nullable = true)
 |-- precio_maximo_historico: double (nullable = true)
 |-- precio_minimo_historico: double (nullable = true)
 |-- modelo: string (nullable = true)
 |-- silhouette: double (nullable = true)

Verificando productos disponibles para filtrar:
+--------------------+
|producto            |
+--------------------+
|Zanahoria           |
|Papa Amarilla       |
|Ajo Criollo O Napuri|
|Papaya              |
|Camote Morado       |
|Platano Bellaco     |
+--------------------+



In [44]:
# Exclusion de productos industriales y validacion de metricas de cluster
PRODUCTOS_EXCLUIDOS = [
    "Fideos Molitalia Spaghetti / Tallarin",
    "Leche Fresca Gloria En Bolsa ( 12 Bolsas De Litro)",
    "Huevos Rosados",
    "Pollo Vivo"
]

# Cargar desde STAGING (lo que dejó el Modelo 2), NO desde el destino final
df_clusters_productos = spark.read.parquet(PATH_STAGING_CLUSTERS) \
    .filter(~F.col("producto").isin(PRODUCTOS_EXCLUIDOS))

print("Verificación de integridad para los 6 productos Core:")
df_clusters_productos.select(
    "producto",
    "cluster",
    F.round("precio_promedio_historico", 2).alias("avg_hist"),
    F.round("volatilidad", 4).alias("volat")
).show(truncate=False)

conteo = df_clusters_productos.count()
if conteo == 6:
    print(f"Verificación exitosa: {conteo} productos agrícolas listos para exportación.")
else:
    print(f"Advertencia: Se detectaron {conteo} productos. Revisa la lista de exclusión.")

# Guardar en el destino FINAL (distinto del origen -> sin colisión de lectura/escritura)
df_clusters_productos.write.mode("overwrite").parquet(PATH_ORO_CLUSTERS)
print(f"Zona Oro actualizada exitosamente en: {PATH_ORO_CLUSTERS}")

Verificación de integridad para los 6 productos Core:
+--------------------+-------+--------+------+
|producto            |cluster|avg_hist|volat |
+--------------------+-------+--------+------+
|Zanahoria           |0      |0.79    |0.2963|
|Papa Amarilla       |0      |2.23    |0.8237|
|Ajo Criollo O Napuri|1      |5.43    |3.1713|
|Papaya              |0      |1.63    |0.3495|
|Camote Morado       |0      |1.3     |0.4969|
|Platano Bellaco     |0      |1.75    |0.2648|
+--------------------+-------+--------+------+

Verificación exitosa: 6 productos agrícolas listos para exportación.
Zona Oro actualizada exitosamente en: gs://food-price-peru-2025-01/oro/clusters_productos/


In [45]:
# Verificar que la escritura en la ruta final se guardo correctamente
check3 = spark.read.parquet(PATH_ORO_CLUSTERS)

print("Schema final de clusters_productos:")
check3.printSchema()
print(f"Filas verificadas: {check3.count()} (esperado: 6, productos agricolas)")
check3.show(truncate=False)

Schema final de clusters_productos:
root
 |-- producto: string (nullable = true)
 |-- cluster: long (nullable = true)
 |-- precio_promedio_historico: double (nullable = true)
 |-- volatilidad: double (nullable = true)
 |-- precio_maximo_historico: double (nullable = true)
 |-- precio_minimo_historico: double (nullable = true)
 |-- modelo: string (nullable = true)
 |-- silhouette: double (nullable = true)

Filas verificadas: 6 (esperado: 6, productos agricolas)
+--------------------+-------+-------------------------+-------------------+-----------------------+-----------------------+----------+----------+
|producto            |cluster|precio_promedio_historico|volatilidad        |precio_maximo_historico|precio_minimo_historico|modelo    |silhouette|
+--------------------+-------+-------------------------+-------------------+-----------------------+-----------------------+----------+----------+
|Zanahoria           |0      |0.7908235307558725       |0.2963246192135816 |1.8414285714285714

In [49]:
# Checklist final Zona Oro
print("="*55)
print("CHECKLIST ZONA ORO")
print("="*55)
print(f"[OK] kpi_precios        -> {PATH_ORO_PRECIOS} ({check1.count()} filas)")
print(f"[OK] kpi_sentimiento    -> {PATH_ORO_SENTIMIENTO} ({check2.count()} filas)")
print(f"[OK] clusters_productos -> {PATH_ORO_CLUSTERS} ({check3.count()} filas)")
print()
print("Avisar a Andrew: las 3 tablas de Zona Oro estan listas para conectar el dashboard.")

CHECKLIST ZONA ORO
[OK] kpi_precios        -> gs://food-price-peru-2025-01/oro/kpi_precios/ (3132 filas)
[OK] kpi_sentimiento    -> gs://food-price-peru-2025-01/oro/kpi_sentimiento/ (465 filas)
[OK] clusters_productos -> gs://food-price-peru-2025-01/oro/clusters_productos/ (6 filas)

Avisar a Andrew: las 3 tablas de Zona Oro estan listas para conectar el dashboard.
